# L23 BEiT-3 Large ingestion

Keyframes stay on Colab SSD. Resumable vectors go to Drive; the L23 FAISS release is written to `AIC_ARTIFACTS/releases/l23-v1`. Use a GPU runtime.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

!pip -q install faiss-cpu sentencepiece 'setuptools<81' timm torchscale transformers==4.57.6 torchmetrics==0.7.3 tensorboardX
!git clone --depth 1 https://github.com/microsoft/unilm.git /content/unilm
!sed -i 's/from torch._six import inf/from math import inf/' /content/unilm/beit3/utils.py
!mkdir -p /content/work
!wget -q https://github.com/addf400/files/releases/download/beit3/beit3_large_patch16_384_coco_retrieval.pth -O /content/beit3_large_patch16_384_coco_retrieval.pth
!wget -q https://github.com/addf400/files/releases/download/beit3/beit3.spm -O /content/beit3.spm
!wget -q https://aic-data.ledo.io.vn/Keyframes_L23.zip -O /content/work/Keyframes_L23.zip
!wget -q https://aic-data.ledo.io.vn/map-keyframes-aic25-b1.zip -O /content/work/map-keyframes-aic25-b1.zip
!unzip -q /content/work/Keyframes_L23.zip -d /content/work/keyframes
!unzip -q /content/work/map-keyframes-aic25-b1.zip -d /content/work/maps

In [ ]:
import csv
import shutil
import sys
from pathlib import Path

import faiss
import numpy as np
import torch
from PIL import Image
from torchvision import transforms
from transformers import XLMRobertaTokenizer

work = Path("/content/work")
output = Path("/content/drive/MyDrive/AIC_ARTIFACTS")
release_name = "l23-v1"
vectors_dir = output / "workers" / "beit3"
release_dir = output / "releases" / release_name
vectors_dir.mkdir(parents=True, exist_ok=True)
release_dir.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
sys.path.insert(0, "/content/unilm/beit3")
import modeling_finetune
import utils

model = modeling_finetune.beit3_large_patch16_384_retrieval()
utils.load_model_and_may_interpolate(
    "/content/beit3_large_patch16_384_coco_retrieval.pth", model, "model|module", ""
)
model = model.to(device).eval()
transform = transforms.Compose([
    transforms.Resize((384, 384), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
tokenizer = XLMRobertaTokenizer("/content/beit3.spm")

In [ ]:
def embed_images(paths, batch_size=4):
    vectors = []
    for start in range(0, len(paths), batch_size):
        images = []
        for path in paths[start : start + batch_size]:
            with Image.open(path) as image:
                images.append(transform(image.convert("RGB")))
        with torch.inference_mode():
            vision, _ = model(image=torch.stack(images).to(device), only_infer=True)
        vectors.append(vision.cpu().numpy().astype("float32"))
    return np.vstack(vectors)

frame_dirs = []
for path in work.rglob("L23_V*"):
    if path.is_dir() and any(path.glob("*.jpg")):
        frame_dirs.append(path)
frame_dirs.sort(key=lambda path: path.name)

for frame_dir in frame_dirs:
    video_id = frame_dir.name
    vector_path = vectors_dir / f"{video_id}.npy"
    if vector_path.exists():
        print(f"skip {video_id}")
        continue
    frames = sorted(frame_dir.glob("*.jpg"), key=lambda path: int(path.stem))
    np.save(vector_path, embed_images(frames))
    print(f"saved {video_id} {len(frames)} frames")

In [ ]:
frame_dirs_by_video = {path.name: path for path in frame_dirs}
rows = []
index = None
next_id = 0

for vector_path in sorted(vectors_dir.glob("L23_V*.npy")):
    video_id = vector_path.stem
    frames = sorted(frame_dirs_by_video[video_id].glob("*.jpg"), key=lambda path: int(path.stem))
    vectors = np.load(vector_path, allow_pickle=False).astype("float32")
    if vectors.ndim != 2 or vectors.shape[1] != 1024:
        raise ValueError(f"{video_id}: expected 1024D BEiT-3 Large vectors, got {vectors.shape}")
    if len(vectors) != len(frames):
        raise ValueError(f"{video_id}: feature/keyframe mismatch")

    mapping_path = next(work.rglob(f"{video_id}.csv"))
    timestamps = {}
    with mapping_path.open(newline="", encoding="utf-8") as file:
        for row in csv.DictReader(file):
            timestamps[int(row["n"])] = float(row["pts_time"])

    faiss.normalize_L2(vectors)
    if index is None:
        index = faiss.IndexIDMap2(faiss.IndexFlatIP(1024))
    ids = np.arange(next_id, next_id + len(frames), dtype="int64")
    index.add_with_ids(vectors, ids)
    for frame, frame_id in zip(frames, ids):
        keyframe_n = int(frame.stem)
        rows.append({
            "frame_id": int(frame_id),
            "frame_uid": f"{video_id}__kf_{keyframe_n}",
            "video_id": video_id,
            "keyframe_n": keyframe_n,
            "timestamp_sec": timestamps[keyframe_n],
            "frame_path": f"keyframes/keyframes/{video_id}/{frame.name}",
            "source": "organizer",
        })
    next_id += len(frames)

if index is None or len(rows) != index.ntotal:
    raise ValueError("could not build a complete L23 index")
faiss.write_index(index, str(release_dir / "beit3.faiss"))
with (release_dir / "frames.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
print(f"indexed {index.ntotal} L23 keyframes")
shutil.rmtree(work)

In [ ]:
query = "a person riding a bicycle"
tokens = tokenizer(query, max_length=64, padding="max_length", truncation=True, return_tensors="pt")["input_ids"].to(device)
with torch.inference_mode():
    _, text_vector = model(text_description=tokens, padding_mask=tokens.eq(tokenizer.pad_token_id), only_infer=True)
text_vector = text_vector.cpu().numpy().astype("float32")
faiss.normalize_L2(text_vector)
scores, ids = index.search(text_vector, 5)
for score, frame_id in zip(scores[0], ids[0]):
    row = rows[int(frame_id)]
    print(row["video_id"], row["timestamp_sec"], round(float(score), 4))